# Arize Singapore Workshop: Trace a LangGraph Agent

In this notebook you will:
1. Build a **customer-support agent** for a fictional retailer (Sunrise Outfitters) with **LangGraph** + **OpenAI**.
2. **Trace** it into **Arize** with one-line auto-instrumentation.
3. Chat with it through a **Gradio** UI (inline, right here in Colab) and watch the traces flow in.

**The only input you need to start is an OpenAI API key.** Tracing later also needs an Arize Space ID + API key (free at [app.arize.com](https://app.arize.com)).

## Step 1 - Install dependencies

In [ ]:
%pip install -q -U \
    langgraph \
    langchain \
    langchain-openai \
    arize-otel \
    openinference-instrumentation-langchain \
    gradio \
    openai

## Step 2 - Add your OpenAI API key

This is the only credential needed to get the agent running.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Step 3 - Define the support tools

Mock tools backed by in-memory data so we need no real backend. The agent will call these to look up orders, check refund eligibility, search the FAQ, and escalate to a human.

In [ ]:
from langchain_core.tools import tool

_ORDERS = {
    "A1001": {"status": "shipped", "item": "Trailblazer Rain Jacket (M, Forest Green)",
               "carrier": "DHL", "tracking": "DHL-SG-77123", "ordered_days_ago": 3,
               "delivered": False, "price_usd": 129.00},
    "A1002": {"status": "delivered", "item": "Summit Hiking Boots (US 9)",
               "carrier": "SingPost", "tracking": "SP-99812", "ordered_days_ago": 20,
               "delivered": True, "price_usd": 159.00},
    "A1003": {"status": "processing", "item": "Coastline Windbreaker (L, Navy)",
               "carrier": None, "tracking": None, "ordered_days_ago": 1,
               "delivered": False, "price_usd": 89.00},
}

_FAQ = [
    (["shipping", "ship", "delivery", "deliver", "how long", "arrive"],
     "Standard shipping within Singapore takes 2-4 business days. International orders take 7-14 business days. You get a tracking number by email once it ships."),
    (["return", "returns", "refund", "exchange", "money back"],
     "You can return unworn items within 30 days of delivery for a full refund. Items must have original tags."),
    (["size", "sizing", "fit", "measurements", "chart"],
     "Jackets run true to size; boots run about half a size large. A size chart is on every product page under 'Size & Fit'."),
    (["payment", "pay", "card", "paynow", "installment"],
     "We accept all major credit cards, PayNow, and Atome installments. Payment is charged when your order ships."),
]


@tool
def lookup_order(order_id: str) -> str:
    """Look up the status and details of a customer order by its ID (e.g. 'A1001')."""
    order = _ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}'. Ask the customer to double-check it."
    parts = [f"Order {order_id.upper()}: {order['item']}", f"Status: {order['status']}",
             f"Placed: {order['ordered_days_ago']} day(s) ago", f"Price: ${order['price_usd']:.2f}"]
    if order["tracking"]:
        parts.append(f"Carrier: {order['carrier']} (tracking {order['tracking']})")
    return ". ".join(parts) + "."


@tool
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order is eligible for a refund (delivered items: within 30 days)."""
    order = _ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}', so eligibility cannot be checked."
    if order["status"] == "processing":
        return f"Order {order_id.upper()} is still processing and can be cancelled now for a full refund."
    if order["delivered"]:
        if order["ordered_days_ago"] <= 30:
            return f"Order {order_id.upper()} was delivered and is within the 30-day window, so it IS eligible for a full refund."
        return f"Order {order_id.upper()} was delivered over 30 days ago; NOT eligible for a standard refund. Offer store credit."
    return f"Order {order_id.upper()} has shipped but not yet delivered. The customer can return it within 30 days of arrival."


@tool
def search_faq(query: str) -> str:
    """Search the help center for shipping, returns, sizing, and payment info."""
    q = query.lower()
    for keywords, answer in _FAQ:
        if any(k in q for k in keywords):
            return answer
    return "No exact FAQ match. We ship in 2-4 business days locally, accept returns within 30 days, and sizing runs true to size."


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate to a human agent and open a support ticket. Use when the customer is upset or the issue is complex."""
    ticket_id = f"TCK-{abs(hash(reason)) % 90000 + 10000}"
    return f"Escalated to a human agent. Ticket {ticket_id} created: '{reason}'. A specialist replies within 24 hours."


SUPPORT_TOOLS = [lookup_order, check_refund_eligibility, search_faq, escalate_to_human]
print(f"Defined {len(SUPPORT_TOOLS)} tools.")

## Step 3b - Build the LangGraph agent

We use LangGraph's prebuilt ReAct agent: the LLM reasons, calls tools, sees the results, and replies.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    "You are Sunny, the customer-support assistant for Sunrise Outfitters, an online "
    "outdoor-apparel retailer in Singapore. Be warm and concise. Use the tools to look "
    "up real order details before answering; never invent statuses, tracking, or prices. "
    "Check eligibility before promising refunds. Escalate to a human if the customer is "
    "upset or you cannot resolve the issue. Keep replies under ~120 words."
)

def build_agent(model="gpt-4o-mini", temperature=0.0):
    llm = ChatOpenAI(model=model, temperature=temperature)
    return create_react_agent(llm, tools=SUPPORT_TOOLS, prompt=SYSTEM_PROMPT)

def run_agent(agent, user_message):
    result = agent.invoke({"messages": [HumanMessage(content=user_message)]})
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and m.content:
            return m.content
    return result["messages"][-1].content

agent = build_agent()
print("Agent ready.")

### Run the agent once (no tracing yet)

In [ ]:
print(run_agent(agent, "Where is my order A1001?"))

## Step 4 - Add Arize tracing

Now we register the Arize tracer and instrument LangChain. LangGraph runs on LangChain runnables, so this single instrumentor captures the **whole graph**: agent reasoning, every LLM call, and every tool call.

Get your **Space ID** and **API key** from [app.arize.com](https://app.arize.com) -> Settings -> Space API Keys.

In [ ]:
from arize.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

os.environ["ARIZE_SPACE_ID"] = getpass("Arize Space ID: ")
os.environ["ARIZE_API_KEY"] = getpass("Arize API key: ")

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="arize-singapore-workshop",
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
print("Tracing enabled -> project 'arize-singapore-workshop'.")

### Re-run the agent - now it's traced

Run a few queries, then open the `arize-singapore-workshop` project in Arize to see the traces (with tool calls and their inputs/outputs).

In [ ]:
for q in [
    "Can I get a refund on order A1002?",
    "How long does shipping take to Singapore?",
    "Order A1003 still hasn't shipped and I'm frustrated!",
]:
    print(f"Q: {q}")
    print(f"A: {run_agent(agent, q)}\n")

## Step 5 - Chat with the agent in a Gradio UI

This launches an interactive chat UI right inside the notebook (and prints a public share link). Every message is traced into Arize.

In [ ]:
import gradio as gr

EXAMPLES = [
    "Where is my order A1001?",
    "Can I get a refund on order A1002?",
    "How long does shipping take to Singapore?",
    "Order A1003 hasn't arrived and I'm frustrated. Help!",
]

# Apple-inspired theme: system font stack, soft background, blue accent, pill buttons.
THEME = gr.themes.Soft(
    primary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.gray,
    radius_size=gr.themes.sizes.radius_lg,
    font=["-apple-system", "BlinkMacSystemFont", "SF Pro Text", "Inter", "system-ui", "sans-serif"],
).set(
    body_background_fill="#f5f5f7",
    block_background_fill="#ffffff",
    block_border_width="0px",
    block_shadow="0 8px 30px rgba(0, 0, 0, 0.06)",
    button_primary_background_fill="#0071e3",
    button_primary_background_fill_hover="#0077ed",
    button_primary_text_color="#ffffff",
    button_large_radius="980px",
    button_small_radius="980px",
    input_radius="22px",
)

CSS = """
.gradio-container {max-width: 820px !important; margin: 0 auto !important;}
#app-header {text-align: center; padding: 22px 0 4px;}
#app-header .logo {font-size: 30px; line-height: 1;}
#app-header h1 {font-weight: 600; letter-spacing: -0.02em; margin: 8px 0 0; font-size: 26px; color: #1d1d1f;}
#app-header p {color: #6e6e73; margin: 6px 0 0; font-size: 15px;}
div[class*="message"] {border-radius: 20px !important; border: none !important; box-shadow: none !important;}
footer {display: none !important;}
.send-btn {min-width: 92px !important;}
"""


def on_submit(message, history):
    if not message or not message.strip():
        return "", history or []
    return "", (history or []) + [{"role": "user", "content": message}]


def on_reply(history):
    try:
        reply = run_agent(agent, history[-1]["content"])
    except Exception as e:
        reply = f"Sorry, something went wrong: {e}"
    return history + [{"role": "assistant", "content": reply}]


with gr.Blocks(title="Sunrise Outfitters Support", fill_height=True) as demo:
    gr.HTML(
        '<div id="app-header"><div class="logo">\u26f0\ufe0f</div>'
        "<h1>Sunrise Outfitters</h1>"
        "<p>AI support assistant - ask about orders, shipping, returns &amp; sizing</p></div>"
    )
    chatbot = gr.Chatbot(show_label=False, height=460,
                         placeholder="<div style='color:#86868b'>Say hello to Sunny \U0001f44b</div>")
    with gr.Row():
        msg = gr.Textbox(show_label=False, placeholder="Message Sunny...", autofocus=True, scale=8, container=False)
        send = gr.Button("Send", variant="primary", scale=0, elem_classes="send-btn")
    gr.Examples(examples=EXAMPLES, inputs=msg, label="Try one")
    clear = gr.Button("Clear conversation", variant="secondary", size="sm")

    msg.submit(on_submit, [msg, chatbot], [msg, chatbot]).then(on_reply, chatbot, chatbot)
    send.click(on_submit, [msg, chatbot], [msg, chatbot]).then(on_reply, chatbot, chatbot)
    clear.click(lambda: [], None, chatbot)

demo.launch(share=True, theme=THEME, css=CSS, debug=False)

## Recap

You built a LangGraph customer-support agent, traced it into Arize with a single instrumentor, and chatted with it through a polished Gradio UI. Every message now appears as a trace (LLM calls + tool calls) in the `arize-singapore-workshop` project.

Repo with the same code + a local Gradio app: **github.com/hakantekgul/arize-singapore-workshop**